# 13. Two pointers

After this notebook you'll be able to walk through a list with two positions at once: from both ends towards the middle, or both from the start at different speeds. You'll also merge two sorted lists.

This isn't new Python. It's a **technique**: a way of using the lists, `while` loops and indexes you already know. Many easy LeetCode problems are two pointers in disguise, so this notebook goes slowly and shows every step.

## Two fingers on a row of cards

Lay a row of cards on a table. Put one finger on the first card and one on the last. Compare the two cards, do something, then move one or both fingers inwards. Stop when the fingers meet.

A **pointer** here is a variable that holds an index. It "points" at one box of the list. **Two pointers** means two such variables, moving through the same list.

```
   index:    0     1     2     3     4
          [ "a", "b", "c", "d", "e" ]
             ↑                 ↑
           left              right
```

`left` starts at `0`, the first index. `right` starts at `len(nums) - 1`, the last index.

In [ ]:
nums = ["a", "b", "c", "d", "e"]
left = 0
right = len(nums) - 1
print(left, right)
print(nums[left], nums[right])

```
0 4
a e
```

## A swap, first

Most two-pointer problems swap two items. You met the swap with tuples in notebook 12: `a, b = b, a`. It works on list boxes too.

In [ ]:
nums = ["a", "b", "c", "d", "e"]
nums[0], nums[4] = nums[4], nums[0]
print(nums)

```
['e', 'b', 'c', 'd', 'a']
```

Python first reads the right side, `(nums[4], nums[0])`, which is `("e", "a")`. Then it puts `"e"` into box 0 and `"a"` into box 4. No helper variable needed.

## From both ends: reverse a list in place

Swap the first and last items, then the second and second-last, and so on, until the fingers meet. **In place** means you change the list you were given. You don't make a new one.

The `print` inside the loop is only there so you can watch the pointers move. Run it.

In [ ]:
def reverse_in_place(nums):
    left = 0
    right = len(nums) - 1
    while left < right:
        print("left =", left, " right =", right, " swap", nums[left], "and", nums[right])
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1
    print("stop: left =", left, " right =", right)

letters = ["a", "b", "c", "d", "e"]
reverse_in_place(letters)
print(letters)

```
left = 0  right = 4  swap a and e
left = 1  right = 3  swap b and d
stop: left = 2  right = 2
['e', 'd', 'c', 'b', 'a']
```

Step by step:

| Step | `left` | `right` | `left < right`? | What happens | List after |
| --- | --- | --- | --- | --- | --- |
| 1 | 0 | 4 | yes | swap `a` and `e` | `[e, b, c, d, a]` |
| 2 | 1 | 3 | yes | swap `b` and `d` | `[e, d, c, b, a]` |
| 3 | 2 | 2 | **no** | stop | |

The middle item `c` never moves. It's already in the right place.

Notice the function doesn't `return` anything. It changed `letters` itself, because a function that gets a list gets **the same** list, not a copy (notebook 07).

**Try it:** change `letters` to `["a", "b", "c", "d"]` (an even number of items) and run it again. How many swaps now?

## Why `left < right`, and not `left != right`?

With 5 items the pointers land on the same box (2 and 2). With 4 items they never land on the same box. They **cross**:

| Step | `left` | `right` | What happens |
| --- | --- | --- | --- |
| 1 | 0 | 3 | swap |
| 2 | 1 | 2 | swap |
| 3 | 2 | 1 | `left` has passed `right`: stop |

`left < right` stops in both cases: when they meet, and when they cross.

`left != right` only stops when they meet. Predict what happens with 4 items, then run the cell below.

In [ ]:
def reverse_wrong(nums):
    left = 0
    right = len(nums) - 1
    while left != right:
        print("left =", left, " right =", right)
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1

letters = ["a", "b", "c", "d"]
reverse_wrong(letters)

```
left = 0  right = 3
left = 1  right = 2
left = 2  right = 1
left = 3  right = 0
left = 4  right = -1
...
IndexError: list index out of range
```

The pointers crossed at step 3 and kept going. They swapped everything back again, then `left` walked off the end of the list. So the rule for "from both ends" is **`while left < right`**.

## From both ends: is it a palindrome?

A **palindrome** reads the same forwards and backwards, like `"level"` or `"racecar"`. Compare the two ends. If they ever differ, it's not a palindrome, and you can stop at once with `return False`.

In [ ]:
def is_palindrome(word):
    left = 0
    right = len(word) - 1
    while left < right:
        print("compare", word[left], "and", word[right])
        if word[left] != word[right]:
            return False
        left += 1
        right -= 1
    return True

print(is_palindrome("racecar"))
print()
print(is_palindrome("rocket"))

```
compare r and r
compare a and a
compare c and c
True

compare r and t
False
```

| Word | Step | `left` | `right` | Letters | Result |
| --- | --- | --- | --- | --- | --- |
| racecar | 1 | 0 | 6 | `r`, `r` | same, move in |
| | 2 | 1 | 5 | `a`, `a` | same, move in |
| | 3 | 2 | 4 | `c`, `c` | same, move in |
| | 4 | 3 | 3 | | fingers met: `True` |
| rocket | 1 | 0 | 5 | `r`, `t` | different: `False` at once |

For `"rocket"` the loop ran only once. That's the point of two pointers: it can stop early.

(`word == word[::-1]` gives the same answer in one line. It's good to know both. The two-pointer way bends to fit harder problems, like skipping spaces, which you'll do in the tasks.)

**Predict, then run:** how many times will `compare` print for `"noon"`? And for `"abcdef"`?

In [ ]:
print(is_palindrome("noon"))
print(is_palindrome("abcdef"))

```
compare n and n
compare o and o
True
compare a and f
False
```

## Slow and fast: a read pointer and a write pointer

Now both pointers start at the **left**, and they move at different speeds.

- The **read** pointer looks at every item, one by one.
- The **write** pointer marks the next box to fill with an item you want to **keep**. It only moves when you keep something.

Think of tidying a bookshelf. One hand picks up each book in turn (read). The other hand marks where the next good book goes (write). Books you keep get packed to the front, in the same order.

Example: keep only the marks of students who passed (50 or more), in place, and return how many you kept.

In [ ]:
def keep_passes(marks):
    write = 0
    for read in range(len(marks)):
        if marks[read] >= 50:
            marks[write] = marks[read]
            write += 1
        print("read =", read, " write =", write, " ", marks)
    return write

marks = [72, 31, 55, 40, 90]
k = keep_passes(marks)
print()
print(k)
print(marks[:k])

```
read = 0  write = 1   [72, 31, 55, 40, 90]
read = 1  write = 1   [72, 31, 55, 40, 90]
read = 2  write = 2   [72, 55, 55, 40, 90]
read = 3  write = 2   [72, 55, 55, 40, 90]
read = 4  write = 3   [72, 55, 90, 40, 90]

3
[72, 55, 90]
```

| `read` | `marks[read]` | Keep? | What happens | `write` after |
| --- | --- | --- | --- | --- |
| 0 | 72 | yes | copy 72 into box 0 | 1 |
| 1 | 31 | no | only `read` moves | 1 |
| 2 | 55 | yes | copy 55 into box 1 | 2 |
| 3 | 40 | no | only `read` moves | 2 |
| 4 | 90 | yes | copy 90 into box 2 | 3 |

At the end, `write` is **how many you kept**, and the first `write` boxes hold them. The boxes after that (`40, 90`) are leftovers. They don't matter.

This one shape solves a whole family of problems. Only two things change:

1. **What you keep.** Here it's "50 or more". It could be "not zero", or "different from the last one I kept".
2. **What you do after the loop** with the leftover boxes. Sometimes nothing. Sometimes you fill them with zeros.

**Try it:** change the rule to keep only the marks **below** 50, and run it again.

### Keep it if it's different from the last one kept

In a **sorted** list, copies of the same value sit next to each other: `[1, 1, 2, 3, 3, 3]`. To keep one of each, keep an item only when it's different from the last item you kept. The last item you kept is in box `write - 1`.

In [ ]:
def keep_one_of_each(nums):
    write = 1                     # box 0 is always kept
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:
            nums[write] = nums[read]
            write += 1
        print("read =", read, " write =", write, " ", nums)
    return write

nums = [1, 1, 2, 3, 3, 3]
k = keep_one_of_each(nums)
print(nums[:k])

```
read = 1  write = 1   [1, 1, 2, 3, 3, 3]
read = 2  write = 2   [1, 2, 2, 3, 3, 3]
read = 3  write = 3   [1, 2, 3, 3, 3, 3]
read = 4  write = 3   [1, 2, 3, 3, 3, 3]
read = 5  write = 3   [1, 2, 3, 3, 3, 3]
[1, 2, 3]
```

The first item is always kept, so both pointers start at 1. Then `read` compares with `nums[write - 1]`, the last kept item.

## One pointer in each list: merge two sorted lists

Two queues at the cinema, each already sorted by ticket number. You want one sorted queue. Look at the **front** of each queue, let the smaller one through, and repeat.

Here there's one pointer in **each** list: `i` for `a`, `j` for `b`.

In [ ]:
def merge(a, b):
    result = []
    i = 0
    j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
        print("i =", i, " j =", j, " ", result)
    # one list is used up; the rest of the other one is already sorted
    result += a[i:]
    result += b[j:]
    return result

print(merge([1, 4, 9], [2, 3, 10, 12]))

```
i = 1  j = 0   [1]
i = 1  j = 1   [1, 2]
i = 1  j = 2   [1, 2, 3]
i = 2  j = 2   [1, 2, 3, 4]
i = 3  j = 2   [1, 2, 3, 4, 9]
[1, 2, 3, 4, 9, 10, 12]
```

| `a[i]` | `b[j]` | Smaller | Who moves |
| --- | --- | --- | --- |
| 1 | 2 | 1 | `i` |
| 4 | 2 | 2 | `j` |
| 4 | 3 | 3 | `j` |
| 4 | 10 | 4 | `i` |
| 9 | 10 | 9 | `i` |
| (none left) | 10 | | loop stops, add `[10, 12]` |

The loop stops when **one** list is used up. Whatever is left in the other list is bigger than everything so far and already sorted, so it goes on the end. `a[i:]` is empty here, and `b[j:]` is `[10, 12]`.

## Common mistakes

**Starting `right` at `len(nums)`.** The last index is `len(nums) - 1`. Run this:

In [ ]:
nums = [3, 1, 4]
right = len(nums)
print(nums[right])

```
IndexError: list index out of range
```

Change `len(nums)` to `len(nums) - 1` and run it again.

**Forgetting to move a pointer.** If no line inside the `while` changes `left` or `right`, the condition never changes and the loop never ends. The cell keeps showing `[*]` on its left. Press the **■ stop button** in the toolbar (or **Kernel → Interrupt Kernel**), then check that every path through the loop moves at least one pointer.

**Moving `write` when you skip.** In the read/write pattern, `write += 1` belongs **inside** the `if`, next to the copy. If it's outside, `write` moves on every item and nothing gets packed.

---

# ✏️ Your turn

Each task has a cell for your code and a check cell under it. Write your code, run your cell, then run the check cell. Each line prints ✓ or ✗.

Use `print` inside your loops while you work, like the examples above. Take it out when it's right.

Stuck? Draw the list on paper and move two fingers along it.

### Task 1: swap the ends

Write `swap_ends(nums)`. It swaps the first and the last item of `nums`, in place. It doesn't return anything.

`[1, 2, 3, 4]` becomes `[4, 2, 3, 1]`.

In [ ]:
def swap_ends(nums):
    # your code here
    pass

In [ ]:
from checks import check

a = [1, 2, 3, 4]
swap_ends(a)
check(a, [4, 2, 3, 1])

b = ["tea", "milk"]
swap_ends(b)
check(b, ["milk", "tea"])

c = [7]
swap_ends(c)
check(c, [7])

### Task 2: palindrome, with two pointers

Write `is_palindrome(word)` using `left` and `right`, not `[::-1]`. Return `True` or `False`.

Think about the edge cases: an empty string `""` and a one-letter word are palindromes.

In [ ]:
def is_palindrome(word):
    # your code here
    pass

In [ ]:
check(is_palindrome("level"), True)
check(is_palindrome("noon"), True)
check(is_palindrome("tea"), False)
check(is_palindrome("abca"), False)
check(is_palindrome("a"), True)
check(is_palindrome(""), True)

### Task 3: count the swaps

Write `count_swaps(n)`. Imagine reversing a list of `n` items with the loop from this notebook. Return how many swaps it does.

Don't work it out with maths. Make `left` and `right` and run the real loop, counting each time round.

| `n` | swaps |
| --- | --- |
| 5 | 2 |
| 4 | 2 |
| 1 | 0 |

In [ ]:
def count_swaps(n):
    # your code here
    pass

In [ ]:
check(count_swaps(5), 2)
check(count_swaps(4), 2)
check(count_swaps(7), 3)
check(count_swaps(1), 0)
check(count_swaps(0), 0)

### Task 4: a pair that adds up

`prices` is a list of prices, **sorted** from smallest to biggest. Write `pair_for(prices, budget)`. It returns the two indexes `[i, j]` (with `i < j`) of two prices that add up to exactly `budget`. There is always exactly one such pair.

Use two pointers from both ends. Look at `prices[left] + prices[right]`:

- Equal to `budget`? Found it.
- Too big? The biggest price is too big for anything left, so move `right` in.
- Too small? Move `left` in, to a bigger price.

`pair_for([10, 20, 35, 50, 75], 70)` gives `[1, 3]`, because `20 + 50` is 70.

In [ ]:
def pair_for(prices, budget):
    # your code here
    pass

In [ ]:
check(pair_for([10, 20, 35, 50, 75], 70), [1, 3])
check(pair_for([1, 2, 3, 4, 5], 9), [3, 4])
check(pair_for([1, 2, 3, 4, 5], 3), [0, 1])
check(pair_for([5, 5], 10), [0, 1])

### Task 5: Reverse String (LeetCode 344)

You get a list of letters `s`. Reverse it **in place**. Don't return anything, and don't use `reverse()` or `[::-1]`.

`["h", "e", "l", "l", "o"]` becomes `["o", "l", "l", "e", "h"]`.

In [ ]:
class Solution:
    def reverseString(self, s: list[str]) -> None:
        # your code here
        pass

In [ ]:
s = ["h", "e", "l", "l", "o"]
Solution().reverseString(s)
check(s, ["o", "l", "l", "e", "h"])

s = ["H", "a", "n", "n", "a", "h"]
Solution().reverseString(s)
check(s, ["h", "a", "n", "n", "a", "H"])

s = ["x"]
Solution().reverseString(s)
check(s, ["x"])

### Task 6: Move Zeroes (LeetCode 283)

Move all the `0`s in `nums` to the end, **in place**, keeping the order of the other numbers. Don't return anything.

`[0, 1, 0, 3, 12]` becomes `[1, 3, 12, 0, 0]`.

Hint: this is the read/write pattern. Keep "not zero". Then, after the loop, fill the leftover boxes (from `write` to the end) with `0`.

In [ ]:
class Solution:
    def moveZeroes(self, nums: list[int]) -> None:
        # your code here
        pass

In [ ]:
nums = [0, 1, 0, 3, 12]
Solution().moveZeroes(nums)
check(nums, [1, 3, 12, 0, 0])

nums = [0]
Solution().moveZeroes(nums)
check(nums, [0])

nums = [4, 2, 0, 0, 7]
Solution().moveZeroes(nums)
check(nums, [4, 2, 7, 0, 0])

nums = [1, 2, 3]
Solution().moveZeroes(nums)
check(nums, [1, 2, 3])

### Task 7: Remove Duplicates from Sorted Array (LeetCode 26)

`nums` is sorted. Keep one copy of each value at the front, in place, and return `k`, how many values you kept. The first `k` boxes must hold them in order. Whatever is after them doesn't matter.

`[0, 0, 1, 1, 1, 2, 2, 3, 3, 4]` gives `k = 5`, and starts with `[0, 1, 2, 3, 4]`.

The check looks at `k` and at `nums[:k]`.

In [ ]:
class Solution:
    def removeDuplicates(self, nums: list[int]) -> int:
        # your code here
        pass

In [ ]:
nums = [1, 1, 2]
k = Solution().removeDuplicates(nums)
check(k, 2)
check(nums[:k], [1, 2])

nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = Solution().removeDuplicates(nums)
check(k, 5)
check(nums[:k], [0, 1, 2, 3, 4])

nums = [7]
k = Solution().removeDuplicates(nums)
check(k, 1)
check(nums[:k], [7])

### Task 8: Squares of a Sorted Array (LeetCode 977)

`nums` is sorted, and it can have negative numbers. Return a **new** list of the squares of each number, also sorted.

`[-4, -1, 0, 3, 10]` gives `[0, 1, 9, 16, 100]`.

This one is a stretch. The easy way is to square everything and `sort()` it, and that's allowed for a first try. Then try the two-pointer way:

- The **biggest** square is always at one of the two ends (a big negative or a big positive).
- Compare `nums[left] * nums[left]` and `nums[right] * nums[right]`. Take the bigger one, move that pointer in.
- You're finding the squares biggest first, so fill `result` from the **back**: make `result = [0] * len(nums)` and a third index `pos` that starts at the last box and goes down by one each time.

In [ ]:
class Solution:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        # your code here
        pass

In [ ]:
check(Solution().sortedSquares([-4, -1, 0, 3, 10]), [0, 1, 9, 16, 100])
check(Solution().sortedSquares([-7, -3, 2, 3, 11]), [4, 9, 9, 49, 121])
check(Solution().sortedSquares([1, 2, 3]), [1, 4, 9])
check(Solution().sortedSquares([-3, -2, -1]), [1, 4, 9])
check(Solution().sortedSquares([5]), [25])

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the cell under it.

**Question 1.** You reverse `[1, 2, 3, 4, 5, 6]` with `while left < right`. How many swaps happen?

- A) 6
- B) 3
- C) 2

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("13-q1", answer_1)

**Question 2.** In the read/write pattern, when does `write` move?

- A) Every time round the loop
- B) Only when an item is kept
- C) Only when an item is skipped

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("13-q2", answer_2)

**Question 3.** Why is `while left != right` wrong for reversing a list?

- A) With an even number of items the pointers cross without ever being equal, so the loop doesn't stop in time
- B) It's slower than `<`
- C) It never swaps the first item

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("13-q3", answer_3)

**Question 4.** You merge `[2, 8]` and `[3, 5, 9, 11]`. The loop stops when `[2, 8]` is used up. What's added at the end?

- A) Nothing
- B) `[3, 5]`
- C) `[9, 11]`

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("13-q4", answer_4)

---

## Done

You can now:

- walk in from both ends with `left` and `right`, and stop with `while left < right`,
- swap items in place with `a, b = b, a`,
- pack the items you keep to the front with a read pointer and a write pointer,
- merge two sorted lists with one pointer in each.

Stuck on a task? The answers are in `solutions/13-two-pointers.ipynb`. Try for real first.

**Next:** a list used as a pile of plates, to match brackets and undo things (`14-stacks`).